In [1]:
# LLM + Embeddings + Loader + Vectorstore
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# RAG / Prompt
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Optional: typing / utils
from typing import List


C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-

Load your documents

In [2]:
# Example: text file
loader = TextLoader("data.txt")
docs = loader.load()

# Optional: PDF
# loader = PyPDFLoader("my_file.pdf")
# docs = loader.load()


Split documents into chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,   # smaller chunks = more precise retrieval
    chunk_overlap=50  # overlap preserves context
)

splits = text_splitter.split_documents(docs)
len(splits)


1

Embeddings + Vector Store

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})  # slightly more results for reranking


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_2924\1262047818.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


LLM

In [5]:
llm = ChatOllama(
    model="llama3",
    temperature=0
)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_2924\1299597105.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


High-quality RAG prompt

In [6]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a highly accurate AI assistant. "
        "Use ONLY the provided context to answer. "
        "If the answer is not in the context, respond 'I don't know.'"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "Context:\n{context}\n\nQuestion:\n{question}"
    ),
])


Build RAG chain

In [7]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
        "chat_history": lambda _: chat_history
    }
    | prompt
    | llm
    | StrOutputParser()
)


Chat memory

In [8]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history: List = []

def chat(question: str):
    answer = rag_chain.invoke(question)

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer


Test improved chat

In [9]:
chat("What is LangChain?")
chat("How does retrieval improve answers?")
chat("Can you summarize LangChain features?")


'According to the provided context, LangChain features include:\n\n1. Prompting\n2. Memory\n3. Retrieval\n4. Agents\n5. More (not specified what "more" refers to)\n\nAdditionally, RAG (Retrieval-Augmented Generation) allows models to answer questions using external documents.'

Inspect retrieved docs

In [10]:
retrieved_docs = retriever.invoke("What is LangChain?")
for doc in retrieved_docs:
    print("----")
    print(doc.page_content)


----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.
----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.
----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.
----
LangChain is a framework for building applications with large language models.
It helps with prompting, memory, retrieval, agents, and more.
RAG allows models to answer questions using external documents.


Hallucination safety test

In [11]:
chat("Who is the president of Mars?")


"I don't know. The provided context doesn't mention anything about the president of Mars or any other information related to that topic. It only talks about LangChain features."